# 03 — Hyperparameter Tuning & Overfitting Check
GridSearchCV with StratifiedKFold(5) to find best params, then train/test F1 comparison to detect overfitting.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd

from src.data import load_data, split_data
from src.tuning import tune_decision_tree, tune_knn, tune_naive_bayes
from src.models import train_decision_tree, train_knn, train_naive_bayes
from src.evaluate import evaluate

In [2]:
X, y = load_data('../data/heart_failure_clinical_records_dataset.csv')
X_train, X_test, y_train, y_test = split_data(X, y)

## Grid Search Results

In [3]:
dt_params, dt_cv_score = tune_decision_tree(X_train, y_train)
knn_params, knn_cv_score = tune_knn(X_train, y_train)
nb_params, nb_cv_score = tune_naive_bayes(X_train, y_train)

tuning_summary = pd.DataFrame([
    {'Model': 'Decision Tree', 'Best Params': str(dt_params), 'CV F1 (weighted)': round(dt_cv_score, 4)},
    {'Model': 'KNN',           'Best Params': str(knn_params), 'CV F1 (weighted)': round(knn_cv_score, 4)},
    {'Model': 'Naive Bayes',   'Best Params': str(nb_params), 'CV F1 (weighted)': round(nb_cv_score, 4)},
]).set_index('Model')

tuning_summary

,Best Params,CV F1 (weighted)
Model,,
Decision Tree,{'max_depth': 4},0.8308
KNN,{'n_neighbors': 7},0.7499
Naive Bayes,{'var_smoothing': 1e-11},0.7566


## Overfitting Check — Train F1 vs Test F1
A large gap between train and test F1 indicates overfitting.

In [6]:
rows = []

for train_fn, name in [
    (train_decision_tree, 'Decision Tree'),
    (train_knn,           'KNN'),
    (train_naive_bayes,   'Naive Bayes'),
]:
    pipe = train_fn(X_train, y_train)
    train_results = evaluate(pipe, X_train, y_train, model_name=f'{name} [train]')
    test_results  = evaluate(pipe, X_test,  y_test,  model_name=f'{name} [test]')
    gap = round(train_results['f1'] - test_results['f1'], 4)
    rows.append({
        'Model':      name,
        'Train F1':   round(train_results['f1'], 4),
        'Test F1':    round(test_results['f1'],  4),
        'Gap':        gap,
        'Overfit?':   'yes' if gap > 0.10 else 'no',
    })
    print()

[Decision Tree [train]] Accuracy: 0.9205 | F1: 0.9187
              precision    recall  f1-score   support

           0       0.91      0.98      0.94       162
           1       0.94      0.81      0.87        77

    accuracy                           0.92       239
   macro avg       0.93      0.89      0.91       239
weighted avg       0.92      0.92      0.92       239

[Decision Tree [test]] Accuracy: 0.7500 | F1: 0.7316
              precision    recall  f1-score   support

           0       0.77      0.90      0.83        41
           1       0.67      0.42      0.52        19

    accuracy                           0.75        60
   macro avg       0.72      0.66      0.67        60
weighted avg       0.74      0.75      0.73        60


[KNN [train]] Accuracy: 0.8285 | F1: 0.8180
              precision    recall  f1-score   support

           0       0.82      0.95      0.88       162
           1       0.85      0.57      0.68        77

    accuracy                  

In [7]:
overfit_df = pd.DataFrame(rows).set_index('Model')
overfit_df

,Train F1,Test F1,Gap,Overfit?
Model,,,,
Decision Tree,0.9187,0.7316,0.1871,yes
KNN,0.8180,0.7002,0.1178,yes
Naive Bayes,0.7690,0.6733,0.0957,no
